# taintwall L1 detector — fine-tune ModernBERT on Colab (free T4)

Fine-tunes `answerdotai/ModernBERT-base` to detect indirect prompt injections in
tool outputs, picks a decision threshold for **FPR ≤ 1%** on the dev split, and
compares against the public baseline `protectai/deberta-v3-base-prompt-injection-v2`.

**Before you run anything:**
1. Runtime → Change runtime type → **T4 GPU**.
2. The dataset (`data/train.jsonl`, `dev.jsonl`, `test.jsonl`) is produced by
   `training/build_dataset.py` and is **gitignored**. Rebuild it on Colab (Cell 2
   clones the repo and runs the builder) or upload your local `data/` folder.
3. The **contamination audit must pass** before training. Cell 3 runs it and stops
   the notebook if any row overlaps AgentDojo. This is the guarantee that the
   AgentDojo evaluation (ring 3) is out-of-distribution.
4. $0: everything here runs on the free tier. Training ~24k rows for 2 epochs is a
   few minutes on a T4.

Nothing is auto-run for you by the session that prepared this notebook — you run it.

## 1. Environment

In [ ]:
# Pinned so a future Colab default doesn't silently change results.
!pip -q install "transformers==4.48.0" "datasets==3.2.0" "accelerate==1.2.1" \
    "scikit-learn==1.6.0" "torch>=2.4"
import torch, transformers, platform
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> T4 GPU"


## 2. Get the code and the dataset

Clone the repo, then **rebuild the data** from the pinned HF seeds (or upload your own `data/`).

In [ ]:
import os, pathlib
REPO = "https://github.com/YuvGev00/taintwall.git"
BRANCH = "feat/l1-dataset"   # change to main once merged
if not pathlib.Path("taintwall").exists():
    !git clone --depth 1 -b $BRANCH $REPO
%cd taintwall
!pip -q install -e . --no-deps   # the taintwall package, for training/ imports

# Rebuild the dataset from the pinned seed datasets (~1 min, streams from HF).
# Skip this cell if you uploaded a data/ folder instead.
if not pathlib.Path("data/train.jsonl").exists():
    !python -m training.build_dataset --target 24000
!wc -l data/*.jsonl


## 3. Contamination gate (must pass)

Stops the notebook if the data shares any 8-gram or attacker entity with AgentDojo. Training on contaminated data would invalidate every downstream number.

In [ ]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "training.contamination_audit",
     "data/train.jsonl", "data/dev.jsonl", "data/test.jsonl",
     "--report", "training/audit_report.json"],
    capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, "CONTAMINATION AUDIT FAILED — do not train on this data."


## 4. Load the splits

In [ ]:
import json
from datasets import Dataset

def load(path):
    rows = [json.loads(l) for l in open(path)]
    return Dataset.from_dict({"text": [r["text"] for r in rows],
                              "label": [int(r["label"]) for r in rows]})

train_ds, dev_ds, test_ds = load("data/train.jsonl"), load("data/dev.jsonl"), load("data/test.jsonl")
print({"train": len(train_ds), "dev": len(dev_ds), "test": len(test_ds)})
print("train positive rate:", sum(train_ds["label"]) / len(train_ds))


## 5. Tokenize

In [ ]:
from transformers import AutoTokenizer
MODEL = "answerdotai/ModernBERT-base"
MAX_LEN = 512   # tool outputs are short; 512 covers the carriers in this dataset
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
dev_tok   = dev_ds.map(tokenize,   batched=True, remove_columns=["text"])
test_tok  = test_ds.map(tokenize,  batched=True, remove_columns=["text"])


## 6. Fine-tune

Two epochs is plenty for this dataset. We keep the best checkpoint by dev AUROC, not accuracy, because the operating point is a low-FPR threshold, not 0.5.

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)
from sklearn.metrics import roc_auc_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
collator = DataCollatorWithPadding(tok)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {"auroc": roc_auc_score(labels, probs)}

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="auroc",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=20261020,
)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=dev_tok,
                  data_collator=collator, compute_metrics=compute_metrics)
trainer.train()


## 7. Choose the threshold on **dev** for FPR ≤ 1%

The threshold is picked on dev and then frozen. We never look at the test split to choose it — that is the whole point of the dev/test separation.

In [ ]:
def probs_for(ds_tok):
    out = trainer.predict(ds_tok)
    return torch.softmax(torch.tensor(out.predictions), dim=-1)[:, 1].numpy(), np.array(out.label_ids)

dev_p, dev_y = probs_for(dev_tok)

TARGET_FPR = 0.01
# Smallest threshold whose FPR on dev negatives is <= TARGET_FPR.
neg = np.sort(dev_p[dev_y == 0])
# The (1 - TARGET_FPR) quantile of negative scores is the lowest threshold meeting the FPR.
threshold = float(np.quantile(neg, 1 - TARGET_FPR))

def rates(p, y, thr):
    pred = (p >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    tn = int(((pred == 0) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    return {"threshold": thr, "TPR": tp / (tp + fn), "FPR": fp / (fp + tn),
            "AUROC": float(roc_auc_score(y, p))}

print("DEV  @ chosen threshold:", json.dumps(rates(dev_p, dev_y, threshold), indent=2))


## 8. Report on the held-out **test** split (threshold frozen from dev)

In [ ]:
test_p, test_y = probs_for(test_tok)
test_metrics = rates(test_p, test_y, threshold)
print("TEST @ frozen threshold:", json.dumps(test_metrics, indent=2))

# Error analysis: carrier types where false positives concentrate (fill the README table).
import collections
test_rows = [json.loads(l) for l in open("data/test.jsonl")]
fp_kinds = collections.Counter(r["carrier_kind"] for r, p in zip(test_rows, test_p)
                               if p >= threshold and r["label"] == 0)
fn_sources = collections.Counter(r["span_source"] for r, p in zip(test_rows, test_p)
                                 if p < threshold and r["label"] == 1)
print("false positives by carrier:", dict(fp_kinds))
print("false negatives by payload source:", dict(fn_sources))


## 9. Baseline: `protectai/deberta-v3-base-prompt-injection-v2`

This is what AgentDojo's built-in `transformers_pi_detector` uses. Same test set, same metric. Report both rows in the README. (The gated `meta-llama/Llama-Prompt-Guard-2-86M` baseline goes here too once access is granted.)

In [ ]:
from transformers import AutoModelForSequenceClassification as AMSC, AutoTokenizer as AT
BASE = "protectai/deberta-v3-base-prompt-injection-v2"
btok = AT.from_pretrained(BASE)
bmodel = AMSC.from_pretrained(BASE).to("cuda").eval()
# Its label 1 ("INJECTION") is the positive class.
inj_idx = [i for i, l in bmodel.config.id2label.items() if "INJ" in l.upper()][0]

@torch.no_grad()
def baseline_probs(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        enc = btok(texts[i:i+bs], truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
        out.append(torch.softmax(bmodel(**enc).logits, -1)[:, inj_idx].cpu().numpy())
    return np.concatenate(out)

base_dev = baseline_probs(dev_ds["text"])
base_thr = float(np.quantile(np.sort(base_dev[dev_y == 0]), 1 - TARGET_FPR))
base_test = baseline_probs(test_ds["text"])
print("BASELINE (protectai) TEST:", json.dumps(rates(base_test, test_y, base_thr), indent=2))
print("\\nComparison at FPR<=1%% (dev-chosen thresholds):")
print(f"  ModernBERT (ours): TPR {test_metrics['TPR']:.3f}, AUROC {test_metrics['AUROC']:.3f}")
print(f"  protectai baseline: TPR {rates(base_test, test_y, base_thr)['TPR']:.3f}, "
      f"AUROC {rates(base_test, test_y, base_thr)['AUROC']:.3f}")


## 10. Save everything needed to serve and reproduce

Push the model to the HF Hub (SPEC: no weights in git). Record the frozen threshold and the metrics next to it.

In [ ]:
import json, pathlib
pathlib.Path("out/best").mkdir(parents=True, exist_ok=True)
trainer.save_model("out/best")
tok.save_pretrained("out/best")
json.dump({"threshold": threshold, "target_fpr": TARGET_FPR,
           "dev": rates(dev_p, dev_y, threshold), "test": test_metrics,
           "model": MODEL, "max_len": MAX_LEN},
          open("out/best/threshold.json", "w"), indent=2)
print(open("out/best/threshold.json").read())

# Optional: push to the Hub (needs `huggingface_hub.login()` with a write token).
# from huggingface_hub import login; login()
# trainer.push_to_hub("YuvGev00/taintwall-l1-modernbert")
# tok.push_to_hub("YuvGev00/taintwall-l1-modernbert")


## Next steps (back in the repo)

- Copy `threshold.json` into the service config; the detector loads the model from
  the Hub and applies this frozen threshold.
- Rings 2 and 3 (SPEC §4): run the saved model over BIPIA / InjecAgent (OOD) and
  end-to-end on AgentDojo. AgentDojo was never in training — the contamination
  audit in Cell 3 is the evidence.
- Fill the README results table and `training/MODEL_CARD.md` with the real numbers.